## Plotting and filtering data

To gain insight in data that you collected or computed, it is essential to not just look at tables of numbers but to also display the data graphically. There are numerous libraries that help you to make nice plots and other visulizations, in this course we first look at Pyplot which is standard tool to quickly visualize data. It works with data contained in lists as well as with numpy arrays. Let's take as a simple example the plotting of an oscillating function $f_1(x)=sin(ax)e^{-b|x|}$ for different values of the parameters $a$ and $b$. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def function_1(x,a=10,b=0.2):
    return (np.sin(a*x)) * np.exp(-b*np.abs(x))

x = np.pi*np.linspace(0,5,500) # Using the linspace function to get x-values
y_1def = function_1(x)         # Call the function to get the standard y-values
y_1mod = function_1(x,5,0.15)  # Call the function again but now with modified parameters

plt.plot(x,y_1def,'r-')        # Plot function_1 with default parameters
plt.plot(x,y_1mod,'b:')        # Plot function_1 with modified parameters
plt.show()                     # Show the plot

**Exercise** *Read the PyPlot tutorial at https://matplotlib.org/stable/tutorials/introductory/pyplot.html until 'Plotting with keyword strings' and modify the cell above to plot as a third line the function $f_2(x)=cos(ax)e^{-b|x|}$ (with a=1 and b=0.3) as a green dashed line.* 

We can also plot data as a scatter plot as is demonstrated with a dataset deposited at https://data.mendeley.com/datasets/yz5rrmvrgd/1 comprising calculated data (DFT level of theory) for all possible diatomic molecules. Here we first read the relevant columns of the csv file that contain the interatomic distance and the bonding energy and plot these.

In [ ]:
# Read dataset by Shibata, Suzuki, Mizoguchi (DOI:10.17632/yz5rrmvrgd.1)
file = 'diatomic_df.csv'
data = np.loadtxt(file, skiprows=1, usecols=(11,12), max_rows=1430, delimiter=',')
plt.xlabel('bond distance in Å')
plt.ylabel('bond energy in eV')
plt.scatter(data[:,0],data[:,1],c='orange',marker='s',s=20)
plt.show()

The plot looks rather ugly because of the orange colour and way too large markers. We can make this nicer by changing the colour and reducing the size of the markers.

**Exercise** *Google the documentation of the scatter plot and change in the above cell the color into dark blue and the marker into something smaller and rounder.*

Now that we can see the data more clearly we note some funny data points in which the "bonding energy" of the molecule is positive rather than negative. This is clearly an artifact of the modelling as bonding energies can only be negative. We therefore want to clean up the dataset by not considering such points. This can be done using the *filtering* options of numpy. We can easily take these out by creating a filter. 

In the cell below we do a number of things:
- We create a 1-dimensional logical array called filter in which we check whether bond energies are negative (as they should be). To see how this array looks like we print it as well.
- We use the filter to get a new array with all rows that pass the test (filter has value True).
- We make as a title a text string in which we print the new dimensions using the *attribute* shape of the new array (this is similar to the size attribute that we saw before, but gives all dimensions as a tuple).
- We plot the array using the standard scatter plot options.

In [ ]:
filter = data[:,1]<0.
print('The filter array is:',filter)
clean_data = data[filter,:]
header = 'Cleaned data: array dimensions are now {}'.format(clean_data.shape)
plt.title(header)
plt.xlabel('bond distance in Å')
plt.ylabel('bond energy in eV')
plt.scatter(clean_data[:,0],clean_data[:,1])
plt.show()

Another thing which is strange with this data set is that some "bond lengths" are very long. According to https://www.webelements.com/periodicity/rad_cov_single/ the longest covalent bond is between two Francium atoms at a little under 4.5 Å. The *brute-force* modelling approach of automatically computing all possible atom pairs apparently leads to failures for some combinations. Let's filter these out.

**Exercise** *Change the cell above such that it does not filter on bond energies but instead leaves out all data with bond lengths > 4.5 Å.* 

Looking again at the bond energy there are also some points that look suspicious given that the bond energy of the strong triple bond in carbonmonoxide is about -11 eV. We better also leave out the outliers that give energies below -15 eV. We want to make a filter that combines these two logical conditions but will see that the intuitive `filter = data[:,1]<0 and data[:,1]>-15.` does *not* work. To see the error message we have written that line below, followed by the *commented out* correct solution. 

The error message points to two useful methods that can be used to see whether there is an element in the array that fulfills a criterion, for instance `np.any(data[:,1]<-15)` which will evaluate to `True` for data but should evaluate to `False` for clean_data. For our purpose this is not what we need, but fortunately there is the method `np.logical_and` that does exactly what we want as this allows us to combine two filter arrays into one. You can see how this works here: https://numpy.org/doc/stable/reference/generated/numpy.logical_and.html. Look in particular at the examples provided on that page and make sure you understand what is happening there. Try modifying and running some of these examples yourself in a code cell to get a thorough understanding.

---
**Exercise** *Filtering*

In [ ]:
filter = (data[:,1]<0.) and (data[:,1]>-15.)
#filter1 = data[:,1]<0.
#filter2 = data[:,1]>-15.
#filter = np.logical_and(filter1,filter2)
clean_data = data[filter,:]
header = 'Cleaned data: array dimensions are now {}'.format(clean_data.shape)
plt.title(header)
plt.xlabel('bond distance in Å')
plt.ylabel('bond energy in eV')
plt.scatter(clean_data[:,0],clean_data[:,1])
plt.show()

*Change the cell above using the following steps:* 
- Uncomment the three lines with correct solutions and comment the line that gave the error (after having looked at this message in the first run).
- Print out the values of `np.any(data[:,1]<-15)` and `np.any(clean_data[:,1]<-15)` to verify that the filtering indeed works.
- Follow the link above to the documentation of np.logical_and and shorten the code by using the shorthand notation with `&` instead of writing np.logical_and(filter1,filter2).
- With the shorthand notation you can easily combine conditions, so use this to filter out also the data with bond lengths above 4.5 Å, like you did in the previous exercise.

Even after filtering we could not observe a clear trend between bond lengths and bond energies. This is mainly due to the fact that simply all atom combinations were tried, combining atoms of varying size. To get meaningful combinations you need more chemical insight and for instance compare just C-C bonds in different molecules. 

---
**Exercise** *Comparing measured and fitted data*

We will leave this dataset, that was merely useful for showing the concept of filtering, and read a data set from file that mimicks a measurement of a concentration of a reactant in a first order reaction. This is suitable for showing the combination of a scatter and a line plot.

In [ ]:
def first_order_decay(x,k=0.2,A_0=1.0):
    return A_0*np.exp(-k*x)

xy = np.load('xy_dataset.npy')
x = xy[0,:]
y_exp = xy[1,:]
y_fit = first_order_decay(x)

plt.scatter(x,y_exp,c='blue',s=1,label='measured')
plt.plot(x,y_fit,'red',label='fit')
plt.xlabel('time in seconds')
plt.ylabel('concentration')
plt.legend()
plt.show()

The fit above is clearly not very good, because the measured decay rate *k* and start concentration $A_0$ are different from the default values in the fit function. You will later in the course learn how you can use the `scipy` library to automatically determine optimal values, for now we will make a best guess by first verifying that the decay is indeed according to first order by plotting the y-data on a logarithmic scale, and then trying out some possible values for *k*. To know how to make logarithmic axes you should first read the section 'Logarithmic and other nonlinear axes' of [the Pyplot tutorial]( https://matplotlib.org/stable/tutorials/introductory/pyplot.html). 

*Make a new code cell to plot ln(y_exp) and ln(y_fit) against x. Then:*
- Determine the start concentration $A_0$ from the xy values that are read in.
- Make a loop to plot (in the same figure) fits for values of k ranging from 0.2 to 0.5 in steps of 0.05.
- Calculate and print in this loop also the *mean absolute deviation* $\frac{1}{N}\sum_x|y_{exp}(x)-y_{fit}(x)|$ with $N$ the number of data points.
- Write in a separate markdown cell what you did and what you think are $A_0$ and $k$.